# Student Dropout Prediction - Exploratory Data Analysis

This notebook explores the student dropout dataset under the current MVP scope. The analysis keeps only Graduate and Dropout records, uses candidate pre-acceptance features, and converts categorical codes into readable labels before visualization.

In [ ]:
# Import libraries used for data loading, transformation, and visualization.
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

This section resolves the project paths, loads the raw dataset, and checks the raw shape before any filtering is applied.

In [ ]:
# Resolve paths so the notebook works from either the project root or the notebooks folder.
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
MVP_READABLE_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mvp_features_readable.csv"
MVP_NUMERIC_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mvp_features_numeric.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())
print("Feature config path:", FEATURE_CONFIG_PATH)
print("Feature config exists:", FEATURE_CONFIG_PATH.exists())
print("Readable MVP data path:", MVP_READABLE_DATA_PATH)

In [ ]:
# Load the raw dataset.
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

In [ ]:
# Run a quick health check on the raw dataset.
print("Raw dataset shape:", df.shape)
print("Raw duplicate rows:", df.duplicated().sum())
print("Raw missing values:", df.isnull().sum().sum())

In [ ]:
# Select Pre-Acceptance MVP Candidate Features
# The MVP scope excludes post-acceptance variables, including daytime/evening attendance,
# semester academic performance variables, macroeconomic indicators, application mode, and application order.
# Categorical values are converted with app/feature_config.json so charts show labels, not raw codes.

post_acceptance_features = [
    "Debtor",
    "Tuition fees up to date",
    "Scholarship holder"
]

academic_semester_features = [
    col for col in df.columns
    if "Curricular units" in col
]

excluded_for_mvp_interpretability = [
    "Mother's occupation",
    "Father's occupation",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
    "Application mode",
    "Application order"
]

candidate_mvp_features = [
    "Marital status",
    "Course",
    "Previous qualification",
    "Mother's qualification",
    "Father's qualification",
    "Displaced",
    "Educational special needs",
    "Gender",
    "Age at enrollment",
    "International"
]

selected_columns = candidate_mvp_features + ["Target"]
missing_selected_columns = [col for col in selected_columns if col not in df.columns]

if missing_selected_columns:
    raise ValueError(f"Missing selected columns: {missing_selected_columns}")

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

value_mappings = feature_config.get("value_mappings", {})


def normalize_category_value(value):
    try:
        numeric_value = float(value)
        if numeric_value.is_integer():
            return str(int(numeric_value))
        return str(numeric_value)
    except (TypeError, ValueError):
        return str(value)


def map_encoded_value(feature, value):
    mapping = value_mappings.get(feature, {})
    key = normalize_category_value(value)
    return mapping.get(key, f"Unknown / unmapped code: {key}")


excluded_columns = [col for col in df.columns if col not in selected_columns]
df_mvp_encoded = df[selected_columns].copy()
df_mvp = df_mvp_encoded.copy()

categorical_display_features = [
    feature for feature in candidate_mvp_features
    if feature != "Age at enrollment"
]

for feature in categorical_display_features:
    df_mvp[feature] = df_mvp[feature].apply(
        lambda value, feature=feature: map_encoded_value(feature, value)
    )

print("Original dataset shape:", df.shape)
print("MVP EDA dataset shape before target filtering:", df_mvp.shape)
print("Selected MVP candidate features:", len(candidate_mvp_features))
print("Excluded columns:", len(excluded_columns))
print("Categorical features converted to text:", len(categorical_display_features))

excluded_columns

In [ ]:
# Preview Readable MVP Dataset
# Keep selected MVP features and original target classes temporarily so the next step can remove Enrolled.
# The processed CSV is saved after Enrolled records are removed.

print("Readable MVP dataset shape before removing Enrolled:", df_mvp.shape)

df_mvp.head()


In [ ]:
# Create Dropout-Graduate EDA Subset
# Remove Enrolled records because this project is framed as binary classification.
# Save both readable and numeric selected-feature datasets after removing Enrolled records.
# Target encoding is intentionally left for the preprocessing notebook.

df_binary = df_mvp[df_mvp["Target"] != "Enrolled"].copy()
df_binary_encoded = df_mvp_encoded[df_mvp_encoded["Target"] != "Enrolled"].copy()

MVP_READABLE_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_binary.to_csv(MVP_READABLE_DATA_PATH, index=False)
df_binary_encoded.to_csv(MVP_NUMERIC_DATA_PATH, index=False)

print("EDA dataset shape before removing Enrolled:", df_mvp.shape)
print("EDA dataset shape after removing Enrolled:", df_binary.shape)
print("Removed Enrolled rows:", df_mvp.shape[0] - df_binary.shape[0])
print("Saved readable MVP dataset to:", MVP_READABLE_DATA_PATH)
print("Saved numeric MVP dataset to:", MVP_NUMERIC_DATA_PATH)
print("Saved readable shape:", df_binary.shape)
print("Saved numeric shape:", df_binary_encoded.shape)

df_binary.head()


## EDA

The EDA is split into non-graphical summaries and visual checks. Non-graphical summaries are used first to understand the filtered dataset structure before plotting.

### Non-graphical EDA

These checks inspect the selected MVP dataset through table summaries, duplicate checks, missing-value checks, feature typing, and category-level summaries.

In [ ]:
# Inspect row count, non-null count, and data types after feature and target filtering.
df_binary.info()

In [ ]:
# Confirm the columns kept in the EDA subset.
df_binary.columns.tolist()

In [ ]:
# Check duplicate rows after narrowing the dataset to MVP candidate features.
# More duplicates are expected here because removed columns may have been the only differences between some records.
duplicate_count = df_binary.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
# Check whether any selected feature contains missing values.
missing_values = df_binary.isnull().sum().sort_values(ascending=False)

print("Total missing values:", missing_values.sum())

missing_values[missing_values > 0]

In [ ]:
# Feature Type Checking
# Many categorical fields are stored as integers, so dtype alone is not enough for visualization decisions.
# Age at enrollment is treated as continuous; the other MVP features are categorical-like.

continuous_features = [
    "Age at enrollment"
]

continuous_features = [
    col for col in continuous_features
    if col in df_binary.columns
]

categorical_like_features = [
    col for col in candidate_mvp_features
    if col not in continuous_features and col in df_binary.columns
]

feature_cardinality = (
    df_binary[candidate_mvp_features]
    .nunique()
    .sort_values(ascending=False)
)

print("Continuous features:", len(continuous_features))
print(continuous_features)

print("\nCategorical-like features:", len(categorical_like_features))
print(categorical_like_features)

feature_cardinality

In [ ]:
# Summarize the continuous feature used in this MVP candidate set.
df_binary[continuous_features].describe().T

In [ ]:
# Categorical Feature Summaries
# Split categorical-like features into low- and high-cardinality groups so each group can be inspected clearly.

low_cardinality_categorical_features = [
    col for col in categorical_like_features
    if df_binary[col].nunique() <= 7
]

high_cardinality_categorical_features = [
    col for col in categorical_like_features
    if df_binary[col].nunique() > 7
]

print("Low-cardinality categorical features:", len(low_cardinality_categorical_features))
print(low_cardinality_categorical_features)

print("\nHigh-cardinality categorical features:", len(high_cardinality_categorical_features))
print(high_cardinality_categorical_features)

In [ ]:
for col in low_cardinality_categorical_features:
    print(f"\nCrosstab: {col} vs Target")
    display(
        pd.crosstab(
            df_binary[col],
            df_binary["Target"],
            normalize="index"
        ).round(3) * 100
    )

In [ ]:
# High-Cardinality Feature Summary
# Show the most frequent categories before looking at category-level dropout-rate differences.

for col in high_cardinality_categorical_features:
    print(f"\nTop categories: {col}")
    display(
        df_binary[col]
        .value_counts()
        .head(10)
        .rename_axis(col)
        .reset_index(name="count")
    )


In [ ]:
# High-Cardinality Feature Dropout Rate
# For features with many categories, dropout rate is often more useful than raw counts alone.
# Keep only categories with at least 30 records to avoid very unstable rates.

MIN_CATEGORY_COUNT = 30

for col in high_cardinality_categorical_features:
    summary = (
        df_binary
        .assign(Target_encoded=df_binary["Target"].map({"Graduate": 0, "Dropout": 1}))
        .groupby(col)
        .agg(
            count=("Target_encoded", "size"),
            dropout_rate=("Target_encoded", "mean")
        )
        .query("count >= @MIN_CATEGORY_COUNT")
        .sort_values("dropout_rate", ascending=False)
        .head(10)
        .reset_index()
    )

    print(f"\nHighest dropout-rate categories: {col}")
    display(summary)

    plt.figure(figsize=(10, 4))
    sns.barplot(
        data=summary,
        x=col,
        y="dropout_rate",
        color="#4C78A8"
    )
    plt.title(f"Top {col} Categories by Dropout Rate")
    plt.xlabel(col)
    plt.ylabel("Dropout Rate")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()


### Graphical EDA

These charts visualize class balance, age distribution, categorical feature counts, and the simple age-target correlation used for exploratory interpretation.

In [ ]:
# Target Distribution
# Check Graduate vs Dropout balance after removing Enrolled records.
# This matters because accuracy can look acceptable even when Dropout recall is weak.

target_percentage = (
    df_binary["Target"]
    .value_counts(normalize=True)
    .loc[["Graduate", "Dropout"]]
    * 100
)

target_percentage_table = pd.DataFrame({
    "percentage": target_percentage.round(2)
})

target_percentage_table

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.barplot(
    x=target_percentage.index,
    y=target_percentage.values
)

plt.title("Target Class Percentage After Removing Enrolled")
plt.xlabel("Target Class")
plt.ylabel("Percentage (%)")
plt.ylim(0, 100)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

plt.show()

##### Target Distribution Insight

After removing Enrolled records, the dataset contains only Graduate and Dropout classes. Graduate is the larger class, but the imbalance is moderate rather than extreme.

Because Dropout is the risk class, later model evaluation should prioritize recall, precision, F1-score, and confusion matrix interpretation instead of relying on accuracy alone.

In [ ]:
# Continuous Features Histogram
# Plot the distribution of the continuous feature.
n_cols = 2
n_rows = int(np.ceil(len(continuous_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))

axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.histplot(
        data=df_binary,
        x=col,
        kde=True,
        ax=axes[i]
    )
    
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Continuous Feature by Target
# Compare the continuous feature between Graduate and Dropout records.
n_cols = 2
n_rows = int(np.ceil(len(continuous_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))

axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.boxplot(
        data=df_binary,
        x="Target",
        y=col,
        order=["Graduate", "Dropout"],
        ax=axes[i]
    )
    
    axes[i].set_title(f"{col} Distribution by Target")
    axes[i].set_xlabel("Target")
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Low-Cardinality Categorical Feature Countplot
# Plot low-cardinality categorical features against the target.
n_cols = 2
n_rows = int(np.ceil(len(low_cardinality_categorical_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))

axes = np.array(axes).flatten()

for i, col in enumerate(low_cardinality_categorical_features):
    sns.countplot(
        data=df_binary,
        x=col,
        hue="Target",
        hue_order=["Graduate", "Dropout"],
        ax=axes[i]
    )
    
    axes[i].set_title(f"{col} Count by Target")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].legend(title="Target")
    axes[i].tick_params(axis="x", rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Continuous Feature Correlation
# Encode the target temporarily only for correlation analysis.
# This is not the final preprocessing target encoding.
corr_data = df_binary[continuous_features].copy()

corr_data["Target_encoded"] = df_binary["Target"].map({
    "Graduate": 0,
    "Dropout": 1
})

continuous_corr = corr_data.corr()

plt.figure(figsize=(6, 4))

sns.heatmap(
    continuous_corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Correlation Heatmap of Age and Target")
plt.show()

# EDA Findings

Based on the exploratory data analysis:

- The final MVP uses 10 fixed enrollment/background features: marital status, course, previous qualification, parents' qualification, displaced status, educational special needs, gender, age at enrollment, and international status.
- `Enrolled` records are removed so the project focuses on binary classification between Graduate and Dropout.
- The readable MVP dataset saved to `data/processed/mvp_features_readable.csv` contains 3630 rows and 11 columns after removing Enrolled.
- `Course` shows strong dropout-rate differences across study programs, including higher-risk categories such as Computer Science, Equinculture, Management Evening Program, Basic Education, and Agronomy.
- `Previous qualification`, `Mother's qualification`, and `Father's qualification` show meaningful category-level differences, so they are kept as early background and academic-path signals.
- `Gender` and `Age at enrollment` show visible differences between Graduate and Dropout groups. These are treated as observed associations, not causal explanations.
- Marital status, displaced status, educational special needs, and international status are kept because they are early, understandable background variables that may add supporting signal.
- Semester academic performance, post-acceptance/admin variables, macroeconomic variables, application mode/order, occupation variables, and nationality are excluded to reduce leakage and keep the app input simple.
- The next stage uses these same 10 features for preprocessing, Logistic Regression, Random Forest, saved artifacts, and the Streamlit app.
